# 🪪 Notebook 3: OAuth 2.0 Authorization Code Flow

**The problem:** the user wants to let *AcmePhoto* see their Google Photos — **without** giving AcmePhoto their Google password.

OAuth solves this with a careful dance between four roles:

- **Resource Owner** — the user.
- **Client** — the app that wants access (AcmePhoto).
- **Authorization Server** — issues access tokens (Google's login screen).
- **Resource Server** — holds the data (the Google Photos API).

We will simulate the **authorization code** flow with two tiny FastAPI apps: one is the auth server, one is the client app. We will run them in the notebook with `httpx` for the HTTP calls.

## Learning objectives
- See the *redirect → consent → code → token → API call* sequence step by step.
- Understand why we exchange a *code* for a *token* (instead of returning the token directly).

## 🛠️ Setup

```bash
cd 01-foundations/authentication-authorization
uv sync
```

This is pure Python; no external services. We use FastAPI's in-process test client so we don't have to start real servers.

In [1]:
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
import secrets, time

# -----------------------------------------------------------------------------
# Authorization Server (think: Google)
# -----------------------------------------------------------------------------
auth_server = FastAPI()

USERS = {"alice": "hunter2"}

# Clients now register: a secret, allowed redirect URIs, and allowed scopes.
CLIENTS = {
    "acmephoto": {
        "secret": "client-secret",
        "redirect_uris": {"https://acmephoto.example/callback"},
        "allowed_scopes": {"photos:read", "photos:write"},
    }
}

CODES = {}           # code -> {user, client_id, scopes, redirect_uri}
TOKENS = {}          # access_token -> {user, scopes, exp}
REFRESH_TOKENS = {}  # refresh_token -> {user, client_id, scopes}

ACCESS_TTL = 2  # seconds — short so the demo can show expiry quickly


def _issue_tokens(user: str, client_id: str, scopes: set[str]):
    access = secrets.token_urlsafe(16)
    refresh = secrets.token_urlsafe(24)
    TOKENS[access] = {"user": user, "scopes": scopes, "exp": time.time() + ACCESS_TTL}
    REFRESH_TOKENS[refresh] = {"user": user, "client_id": client_id, "scopes": scopes}
    return access, refresh


@auth_server.post("/authorize")
def authorize(username: str, password: str, client_id: str,
              redirect_uri: str, scope: str = "", state: str = ""):
    # In a real flow this would be a browser consent screen, not a JSON POST.
    if USERS.get(username) != password:
        raise HTTPException(401, "bad creds")
    client = CLIENTS.get(client_id)
    if not client:
        raise HTTPException(400, "unknown client")
    # Reject redirect URIs that weren't pre-registered — this is critical.
    if redirect_uri not in client["redirect_uris"]:
        raise HTTPException(400, "bad redirect_uri")
    # Only allow scopes the client registered for.
    requested = set(scope.split()) if scope else set()
    if not requested.issubset(client["allowed_scopes"]):
        raise HTTPException(400, "scope not allowed")

    code = secrets.token_urlsafe(8)
    CODES[code] = {
        "user": username,
        "client_id": client_id,
        "scopes": requested,
        "redirect_uri": redirect_uri,
    }
    # The `state` parameter is echoed back verbatim — the client uses it to
    # detect CSRF (it must match what the client sent originally).
    return {"redirect_to": f"{redirect_uri}?code={code}&state={state}"}


@auth_server.post("/token")
def token(grant_type: str = "authorization_code",
          code: str = "", client_id: str = "", client_secret: str = "",
          redirect_uri: str = "", refresh_token: str = ""):
    client = CLIENTS.get(client_id)
    if not client or client["secret"] != client_secret:
        raise HTTPException(401, "bad client")

    if grant_type == "authorization_code":
        info = CODES.pop(code, None)
        if not info or info["client_id"] != client_id:
            raise HTTPException(400, "bad code")
        # redirect_uri must match the one used in /authorize.
        if info["redirect_uri"] != redirect_uri:
            raise HTTPException(400, "redirect_uri mismatch")
        access, refresh = _issue_tokens(info["user"], client_id, info["scopes"])
        return {"access_token": access, "refresh_token": refresh,
                "expires_in": ACCESS_TTL, "scope": " ".join(sorted(info["scopes"]))}

    if grant_type == "refresh_token":
        info = REFRESH_TOKENS.get(refresh_token)
        if not info or info["client_id"] != client_id:
            raise HTTPException(400, "bad refresh")
        access, _ = _issue_tokens(info["user"], client_id, info["scopes"])
        return {"access_token": access, "expires_in": ACCESS_TTL,
                "scope": " ".join(sorted(info["scopes"]))}

    raise HTTPException(400, "unsupported grant_type")


# -----------------------------------------------------------------------------
# Resource Server (think: Google Photos API) — now scope- and expiry-aware
# -----------------------------------------------------------------------------
resource_server = FastAPI()

@resource_server.get("/photos")
def photos(access_token: str):
    info = TOKENS.get(access_token)
    if not info:
        raise HTTPException(401, "bad token")
    if info["exp"] < time.time():
        raise HTTPException(401, "token expired")
    if "photos:read" not in info["scopes"]:
        raise HTTPException(403, "missing scope photos:read")
    return {"user": info["user"], "photos": ["beach.jpg", "cat.jpg"]}


auth_client = TestClient(auth_server)
res_client = TestClient(resource_server)
print("FastAPI apps ready ✅ (with state, scopes, expiry, refresh)")


FastAPI apps ready ✅ (with state, scopes, expiry, refresh)


In [2]:
# -----------------------------------------------------------------------------
# THE FLOW (annotated step by step)
# -----------------------------------------------------------------------------

# The client generates a random `state` BEFORE redirecting. It remembers it
# locally (e.g. in the user's session) so it can verify the callback later.
client_state = secrets.token_urlsafe(8)
print("Client generated state:", client_state)

# Step 1: AcmePhoto sends Alice to the auth server with client_id, redirect_uri,
# scope, and state. Alice logs in there and clicks "Allow".
r = auth_client.post("/authorize", params={
    "username": "alice", "password": "hunter2",
    "client_id": "acmephoto",
    "redirect_uri": "https://acmephoto.example/callback",
    "scope": "photos:read",
    "state": client_state,
})
redirect = r.json()["redirect_to"]
print("Step 1 — auth server redirects browser to:", redirect)

# Step 2: AcmePhoto's backend parses the callback URL.
from urllib.parse import urlparse, parse_qs
qs = parse_qs(urlparse(redirect).query)
code = qs["code"][0]
returned_state = qs["state"][0]

# CRITICAL: verify the state matches what we sent. If not, this could be CSRF.
assert returned_state == client_state, "state mismatch — possible CSRF!"
print("Step 2 — state verified ✅, received code:", code)

# Step 3: AcmePhoto exchanges the code for tokens, proving its identity with
# client_secret. This is a server-to-server back-channel call.
r = auth_client.post("/token", params={
    "grant_type": "authorization_code",
    "code": code,
    "client_id": "acmephoto",
    "client_secret": "client-secret",
    "redirect_uri": "https://acmephoto.example/callback",
})
body = r.json()
access_token = body["access_token"]
refresh_token = body["refresh_token"]
print("Step 3 — got tokens:", body)

# Step 4: AcmePhoto calls the Resource Server with the access token.
r = res_client.get("/photos", params={"access_token": access_token})
print("Step 4 — photos:", r.json())


Client generated state: JwpidpmDZlk
Step 1 — auth server redirects browser to: https://acmephoto.example/callback?code=KwoV7ll-lno&state=JwpidpmDZlk
Step 2 — state verified ✅, received code: KwoV7ll-lno
Step 3 — got tokens: {'access_token': '2mQ2_4sDnQdBJd6Duk8_YA', 'refresh_token': 'GDXQcSFkOd2ht8chq3NWoMHh-pSxqFjQ', 'expires_in': 2, 'scope': 'photos:read'}
Step 4 — photos: {'user': 'alice', 'photos': ['beach.jpg', 'cat.jpg']}


## 🎯 Scopes: principle of least privilege

A scope is a label on the token that says *what it is allowed to do*. `photos:read` lets you list photos; `photos:write` would let you upload. The resource server checks the scope on every request.

Let's see what happens if AcmePhoto asks for a scope the client isn't registered for, and what happens if it tries to hit an endpoint the token doesn't cover.

In [3]:
# 1) Requesting a scope that is not registered for this client → rejected.
r = auth_client.post("/authorize", params={
    "username": "alice", "password": "hunter2",
    "client_id": "acmephoto",
    "redirect_uri": "https://acmephoto.example/callback",
    "scope": "admin:everything",   # not in allowed_scopes
    "state": secrets.token_urlsafe(8),
})
print("unregistered scope →", r.status_code, r.json())

# 2) Requesting a scope but then calling an endpoint that requires a DIFFERENT scope.
#    For this demo we only have /photos which requires photos:read, so asking for
#    photos:write alone should fail.
r = auth_client.post("/authorize", params={
    "username": "alice", "password": "hunter2",
    "client_id": "acmephoto",
    "redirect_uri": "https://acmephoto.example/callback",
    "scope": "photos:write",
    "state": "x",
})
code2 = parse_qs(urlparse(r.json()["redirect_to"]).query)["code"][0]
r = auth_client.post("/token", params={
    "grant_type": "authorization_code", "code": code2,
    "client_id": "acmephoto", "client_secret": "client-secret",
    "redirect_uri": "https://acmephoto.example/callback",
})
narrow_token = r.json()["access_token"]
r = res_client.get("/photos", params={"access_token": narrow_token})
print("wrong-scope request →", r.status_code, r.json())


unregistered scope → 400 {'detail': 'scope not allowed'}
wrong-scope request → 403 {'detail': 'missing scope photos:read'}


## ⏳ Token expiry and the `refresh_token`

Access tokens are deliberately **short-lived** (we set 2 seconds — production is typically 5–60 minutes). When one expires, the client exchanges a **refresh token** for a new access token, *without* bothering the user again. Refresh tokens live longer and are stored only on the client's backend.

Why two tokens? If an access token leaks, the damage window is tiny. Refresh tokens don't fly out on every API call, so they leak far less often.

In [4]:
# Wait for the access token to expire.
print("Calling /photos right now:", res_client.get("/photos", params={"access_token": access_token}).json())
time.sleep(ACCESS_TTL + 0.5)
print("After expiry      :", res_client.get("/photos", params={"access_token": access_token}).json())

# Use the refresh_token to get a fresh access_token — no user interaction needed.
r = auth_client.post("/token", params={
    "grant_type": "refresh_token",
    "refresh_token": refresh_token,
    "client_id": "acmephoto",
    "client_secret": "client-secret",
})
new_access = r.json()["access_token"]
print("Refreshed         :", res_client.get("/photos", params={"access_token": new_access}).json())


Calling /photos right now: {'user': 'alice', 'photos': ['beach.jpg', 'cat.jpg']}


After expiry      : {'detail': 'token expired'}
Refreshed         : {'user': 'alice', 'photos': ['beach.jpg', 'cat.jpg']}


## ✅ Recap

The OAuth dance trades a tiny bit of complexity for a huge security win: **third-party apps can act on behalf of the user without ever seeing the password**, and the user can revoke access at any time.

Key protections we added in this notebook:

- **`state`** — random value the client sends and verifies on callback. Stops CSRF.
- **Redirect URI allow-list** — only pre-registered URIs are accepted.
- **Scopes** — the token carries only the permissions the user consented to.
- **Short-lived access token + refresh token** — limits the damage of a leak.

### 🔑 PKCE (brief)

Mobile apps and single-page apps can't keep a `client_secret` — anyone can unpack the app. **PKCE** ("Proof Key for Code Exchange") replaces the secret with a one-time `code_verifier`/`code_challenge` pair that the client generates for each login. Same flow, no long-lived secret. **All modern OAuth clients should use PKCE**, even on the server side.

### 🆔 OAuth vs OpenID Connect (brief)

OAuth 2.0 is **authorization** ("what can this app do on my behalf?"). **OpenID Connect (OIDC)** is a thin layer on top that adds **authentication** ("who is this user?") by returning an additional **`id_token`** (a signed JWT with the user's identity) alongside the access token. "Log in with Google" buttons are OIDC.
